In [2]:
import os
os.environ['NEO4J_URI'] = 'bolt://localhost:7687'
os.environ['QWEN3_ENDPOINT'] = 'http://localhost:11434'
print('✅ Environment configurato per esecuzione locale.')

✅ Environment configurato per esecuzione locale.


# 🏗️ Legal GraphRAG - Orchestrator

### 1. Gestione Infrastruttura

In [3]:
# Avvia i container e costruisce l'ambiente
!docker-compose up -d --build

#1 [internal] load local bake definitions
#1 reading from stdin 516B 0.0s done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 732B 0.0s done
#2 DONE 0.1s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 1.4s

#4 [internal] load .dockerignore
#4 transferring context: 118B done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 1.22MB 0.1s done
#5 DONE 0.1s

#6 [1/6] FROM docker.io/library/python:3.11-slim@sha256:6d85378d88a19cd4d76079817532d62232be95757cb45945a99fec8e8084b9c2
#6 resolve docker.io/library/python:3.11-slim@sha256:6d85378d88a19cd4d76079817532d62232be95757cb45945a99fec8e8084b9c2 0.1s done
#6 DONE 0.1s

#7 [2/6] WORKDIR /app
#7 CACHED

#8 [3/6] RUN apt-get update && apt-get install -y     gcc     python3-dev     libxml2-dev     libxslt-dev     curl     && rm -rf /var/lib/apt/lists/*
#8 CACHED

#9 [4/6] COPY requirements.txt .
#9 CACHED

#10 [5/6] RUN pip install --no-cache-dir -r r

 ollama Pulling 
 ollama Pulled 
 tesi-app  Built
time="2026-05-03T23:19:00+02:00" level=warning msg="Found orphan containers ([legal-graph-ingestion]) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up."
 Container legal-graph-neo4j  Running
 Container legal-graph-ollama  Running
 Container legal-graph-app  Recreate
 Container legal-graph-app  Recreated
 Container legal-graph-app  Starting
 Container legal-graph-app  Started


In [4]:
# Verifica se Neo4j e Ollama sono pronti
import manage
status = manage.check_health()
print(f"Stato Neo4j: {status['neo4j']}")
print(f"Stato Ollama: {status['ollama']}")

Stato Neo4j: 🟢 Online
Stato Ollama: 🟢 Online


### 2. Ingestione Dati (Scaricamento)

In [4]:
# Scarica Legge Opere Idrauliche (1904)
import manage, asyncio
await manage.run_targeted_ingest("normattiva", {"testo": "idraulica", "annoProvvedimento": 1904})

2026-05-03 22:59:20,623 - manage - INFO - Targeted Ingestion started for normattiva with params: {'testo': 'idraulica', 'annoProvvedimento': 1904}
2026-05-03 22:59:20,625 - src.ingestion.async_normattiva_client - INFO - Initiating search with params: {'testo': 'idraulica', 'annoProvvedimento': 1904}
2026-05-03 22:59:21,580 - src.ingestion.async_normattiva_client - INFO - Search initiated. Token: 72deae6d-a181-4d78-a13d-eb49217d4fde
2026-05-03 22:59:21,851 - src.ingestion.async_normattiva_client - INFO - Search 72deae6d-a181-4d78-a13d-eb49217d4fde confirmed.
2026-05-03 22:59:22,130 - src.ingestion.async_normattiva_client - INFO - Status 1: Ricerca confermata in attesa di elaborazione. Waiting...
2026-05-03 22:59:27,278 - src.ingestion.async_normattiva_client - INFO - Status 1: Ricerca confermata in attesa di elaborazione. Waiting...
2026-05-03 22:59:32,449 - src.ingestion.async_normattiva_client - INFO - Status 1: Ricerca confermata in attesa di elaborazione. Waiting...
2026-05-03 22:59

'data\\raw\\normattiva\\normattiva_export_72deae6d-a181-4d78-a13d-eb49217d4fde.zip'

In [5]:
# Scarica Legge Minori Stranieri (2017)
await manage.run_targeted_ingest("normattiva", {"testo": "minore straniero", "annoProvvedimento": 2017})

2026-05-03 23:01:15,789 - manage - INFO - Targeted Ingestion started for normattiva with params: {'testo': 'minore straniero', 'annoProvvedimento': 2017}
2026-05-03 23:01:15,791 - src.ingestion.async_normattiva_client - INFO - Initiating search with params: {'testo': 'minore straniero', 'annoProvvedimento': 2017}
2026-05-03 23:01:16,803 - src.ingestion.async_normattiva_client - INFO - Search initiated. Token: 3f4057a3-f468-4e0d-ac6f-a5a5d287f9b2
2026-05-03 23:01:17,066 - src.ingestion.async_normattiva_client - INFO - Search 3f4057a3-f468-4e0d-ac6f-a5a5d287f9b2 confirmed.
2026-05-03 23:01:17,317 - src.ingestion.async_normattiva_client - INFO - Status 1: Ricerca confermata in attesa di elaborazione. Waiting...
2026-05-03 23:01:22,483 - src.ingestion.async_normattiva_client - INFO - Status 1: Ricerca confermata in attesa di elaborazione. Waiting...
2026-05-03 23:01:27,614 - src.ingestion.async_normattiva_client - INFO - Status 1: Ricerca confermata in attesa di elaborazione. Waiting...
20

'data\\raw\\normattiva\\normattiva_export_3f4057a3-f468-4e0d-ac6f-a5a5d287f9b2.zip'

### 3. Sincronizzazione Knowledge Graph

In [6]:
# Esegue il parsing degli XML e carica i dati su Neo4j
await manage.run_parse_and_load(
    raw_dir="data/raw/normattiva",
    output_jsonl="data/processed/normattiva.jsonl",
    teseo_rdf="data/external/teseo_sample.rdf"
)

2026-05-03 23:02:19,678 - manage - INFO - Parsing 222 files...
2026-05-03 23:02:19,681 - src.parsing.parser - INFO - Parsing file: data\raw\normattiva\normattiva_export_13d1e3b1-9d45-4f41-b36e-b6bb8c5692c8\DECRETO DEL PRESIDENTE DEL CONSIGLIO DEI MINISTRI_20240304_40\2024-04-02_24G00056_VIGENZA_2026-01-25_V0.xml
2026-05-03 23:02:19,723 - src.parsing.meta_parser - INFO - Generated synthetic URN: urn:nir:stato:legge:2024-03-04;40
2026-05-03 23:02:19,724 - src.parsing.parser - INFO - Fase A completata — URN: urn:nir:stato:legge:2024-03-04;40, Type: Legge, Date: 2024-03-04
2026-05-03 23:02:19,730 - src.parsing.body_parser - INFO - Found attachments section, parsing...
2026-05-03 23:02:19,731 - src.parsing.body_parser - INFO - Body parsed: 33 nodes, 60 edges
2026-05-03 23:02:19,732 - src.parsing.parser - INFO - ✅ Parse complete — 33 nodes, 60 edges
2026-05-03 23:02:19,735 - src.parsing.parser - INFO - Parsing file: data\raw\normattiva\normattiva_export_13d1e3b1-9d45-4f41-b36e-b6bb8c5692c8\D